In [0]:
%pip install newsapi-python vaderSentiment

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from newsapi import NewsApiClient
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from pyspark.sql import Row
from pyspark.sql.types import *
from datetime import datetime

NEWS_API_KEY = "38309e2f905848baacbe0d1741f8d318"

newsapi = NewsApiClient(api_key=NEWS_API_KEY)
analyzer = SentimentIntensityAnalyzer()

print("Ready!")

Ready!


In [0]:
brand_queries = {
    "Tesla": [
        "Tesla Brand",
        "Elon Musk",
        "Tesla Model 3",
        "Tesla Model Y",
        "Tesla Cybertruck"
    ],
    "Hyundai": [
        "Hyundai Ioniq",
        "Hyundai EV",
        "Hyundai Ioniq 5",
        "Hyundai Ioniq 6"
    ],
    "Rivian": [
        "Rivian truck",
        "Rivian R1T",
        "Rivian R1S",
        "Rivian electric"
    ],
    "Ford EV": [
        "Ford Mustang Mach-E",
        "Ford electric vehicle",
        "Ford EV"
    ],
    "BYD": [
        "BYD electric car",
        "BYD EV",
        "BYD Atto",
        "BYD Seal"
    ]
}

In [0]:
all_rows = []
article_counter = 1

for brand, queries in brand_queries.items():
    print(f"\nFetching: {brand}...")
    
    for query_type in queries:
        print(f"  → {query_type}")
        
        response = newsapi.get_everything(
            q=query_type,
            language="en",
            sort_by="publishedAt",
            page_size=100
        )
        
        articles = response["articles"]
        print(f"     {len(articles)} articles found")
        
        for article in articles:
            text = f"{article['title']} {article['description'] or ''}"
            scores = analyzer.polarity_scores(text)
            
            if scores["compound"] >= 0.05:
                label = "positive"
            elif scores["compound"] <= -0.05:
                label = "negative"
            else:
                label = "neutral"
            
            all_rows.append(Row(
                article_id      = str(article_counter),
                brand           = brand,
                query_type      = query_type,
                source          = article["source"]["name"] or "Unknown",
                author          = article["author"] or "Unknown",
                title           = article["title"],
                description     = article["description"] or "",
                url             = article["url"],
                published_at    = datetime.strptime(article["publishedAt"], "%Y-%m-%dT%H:%M:%SZ"),
                collected_at    = datetime.now(),
                compound        = float(scores["compound"]),
                positive_score  = float(scores["pos"]),
                negative_score  = float(scores["neg"]),
                neutral_score   = float(scores["neu"]),
                sentiment_label = label
            ))
            article_counter += 1

print(f"\nTotal articles collected: {len(all_rows)}")


Fetching: Tesla...
  → Tesla Brand
     99 articles found
  → Elon Musk
     100 articles found
  → Tesla Model 3
     98 articles found
  → Tesla Model Y
     98 articles found
  → Tesla Cybertruck
     93 articles found

Fetching: Hyundai...
  → Hyundai Ioniq
     97 articles found
  → Hyundai EV
     98 articles found
  → Hyundai Ioniq 5
     76 articles found
  → Hyundai Ioniq 6
     32 articles found

Fetching: Rivian...
  → Rivian truck
     26 articles found
  → Rivian R1T
     19 articles found
  → Rivian R1S
     22 articles found
  → Rivian electric
     86 articles found

Fetching: Ford EV...
  → Ford Mustang Mach-E
     22 articles found
  → Ford electric vehicle
     98 articles found
  → Ford EV
     97 articles found

Fetching: BYD...
  → BYD electric car
     92 articles found
  → BYD EV
     97 articles found
  → BYD Atto
     43 articles found
  → BYD Seal
     31 articles found

Total articles collected: 1424


In [0]:
schema = StructType([
    StructField("article_id",      StringType(),    False),
    StructField("brand",           StringType(),    False),
    StructField("query_type",      StringType(),    False),
    StructField("source",          StringType(),    True),
    StructField("author",          StringType(),    True),
    StructField("title",           StringType(),    False),
    StructField("description",     StringType(),    True),
    StructField("url",             StringType(),    True),
    StructField("published_at",    TimestampType(), False),
    StructField("collected_at",    TimestampType(), False),
    StructField("compound",        DoubleType(),    True),
    StructField("positive_score",  DoubleType(),    True),
    StructField("negative_score",  DoubleType(),    True),
    StructField("neutral_score",   DoubleType(),    True),
    StructField("sentiment_label", StringType(),    False),
])

df = spark.createDataFrame(all_rows, schema)
df.write.format("delta").mode("overwrite").saveAsTable("tesla_sentiment.articles")

print(f"Written {df.count()} rows to tesla_sentiment.articles")

Written 1424 rows to tesla_sentiment.articles


In [0]:
spark.sql("""
    SELECT
        brand,
        COUNT(*)                                                        AS total_articles,
        SUM(CASE WHEN sentiment_label = 'positive' THEN 1 ELSE 0 END)  AS positive,
        SUM(CASE WHEN sentiment_label = 'negative' THEN 1 ELSE 0 END)  AS negative,
        SUM(CASE WHEN sentiment_label = 'neutral'  THEN 1 ELSE 0 END)  AS neutral,
        ROUND(AVG(compound), 4)                                         AS avg_sentiment
    FROM tesla_sentiment.articles
    GROUP BY brand
    ORDER BY avg_sentiment DESC
""").show()

+-------+--------------+--------+--------+-------+-------------+
|  brand|total_articles|positive|negative|neutral|avg_sentiment|
+-------+--------------+--------+--------+-------+-------------+
|Ford EV|           217|     120|      44|     53|       0.2545|
| Rivian|           153|      87|      39|     27|       0.2156|
|  Tesla|           488|     269|     129|     90|       0.1749|
|    BYD|           263|     142|      75|     46|        0.164|
|Hyundai|           303|     149|      91|     63|       0.1283|
+-------+--------------+--------+--------+-------+-------------+



In [0]:
spark.sql("""
    SELECT
        brand,
        query_type,
        COUNT(*)                AS article_count,
        ROUND(AVG(compound), 4) AS avg_sentiment
    FROM tesla_sentiment.articles
    GROUP BY brand, query_type
    ORDER BY brand, avg_sentiment DESC
""").show(50, truncate=40)

+-------+---------------------+-------------+-------------+
|  brand|           query_type|article_count|avg_sentiment|
+-------+---------------------+-------------+-------------+
|    BYD|             BYD Seal|           31|       0.2441|
|    BYD|               BYD EV|           97|       0.1958|
|    BYD|     BYD electric car|           92|       0.1424|
|    BYD|             BYD Atto|           43|       0.0806|
|Ford EV|Ford electric vehicle|           98|       0.4361|
|Ford EV|  Ford Mustang Mach-E|           22|       0.1082|
|Ford EV|              Ford EV|           97|       0.1042|
|Hyundai|      Hyundai Ioniq 6|           32|       0.2116|
|Hyundai|           Hyundai EV|           98|       0.1907|
|Hyundai|      Hyundai Ioniq 5|           76|       0.0928|
|Hyundai|        Hyundai Ioniq|           97|       0.0654|
| Rivian|           Rivian R1T|           19|       0.3346|
| Rivian|           Rivian R1S|           22|       0.2497|
| Rivian|      Rivian electric|         